# Notebook 23 — Predicting Phase-Lock Quality from the Effective Noise Coordinate

This notebook takes the effective-noise result from Notebook 22b one step further.

Goal:
- fit a simple 1D predictor for the balanced phase-lock order parameter,
- use only the effective coordinate
  `gamma_eff = gamma + lambda * gamma_phi`,
- compare the 1D prediction against the full 2D simulation.

This asks a stronger question than Notebook 22b:

**Can the noisy phase-locked CZ quality be predicted from one reduced coordinate alone, without rerunning the full 2D model everywhere?**

Outputs:
1. fitted effective-noise coordinate,
2. a 1D predictor `S_hat(gamma_eff)`,
3. prediction-vs-truth heatmaps,
4. residual maps,
5. quantitative prediction scores.


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'matplotlib', 'scipy']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.optimize import minimize_scalar
from qutip import basis, qeye, tensor, sigmax, mesolve


## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
n = r * r.dag()
sigma_minus = g * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

basis_states = [gg, gr, rg, rr]

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
n1 = tensor(n, I)
n2 = tensor(I, n)
sm1 = tensor(sigma_minus, I)
sm2 = tensor(I, sigma_minus)

U_cz = np.diag([1, 1, 1, -1]).astype(complex)


## Phase-locked protocol

In [ ]:
opt = {
    'T': 26.0,
    'alpha': 0.10,
    'omega_max': 1.09,
    'delta0': 1.00,
    'p': 1.00,
    'q': 0.72,
}
V = 4.0

for k, v in opt.items():
    print(f"{k}: {v:.6f}")


## Shaped schedules and Hamiltonian

In [ ]:
def omega_shaped(t, T, omega_max, p):
    s = t / T
    base = 16.0 * (s * (1.0 - s)) ** 2
    return omega_max * np.maximum(base, 0.0) ** p

def delta_shaped(t, T, V, alpha, delta0, q):
    s = t / T
    shaped = s ** q
    return delta0 + alpha * V * 0.5 * (1.0 - np.cos(np.pi * shaped))

H_omega = 0.5 * (sx1 + sx2)
H_delta = -(n1 + n2)
H_V = n1 * n2

def build_time_dependent_hamiltonian(T, omega_max, alpha, V, delta0, p, q):
    def coeff_omega(t, args=None):
        return omega_shaped(t, T=T, omega_max=omega_max, p=p)
    def coeff_delta(t, args=None):
        return delta_shaped(t, T=T, V=V, alpha=alpha, delta0=delta0, q=q)
    return [
        [H_omega, coeff_omega],
        [H_delta, coeff_delta],
        [H_V, lambda t, args=None: V],
    ]


## Collapse operators

In [ ]:
def collapse_ops(gamma_decay=0.0, gamma_phi=0.0):
    ops = []
    if gamma_decay > 0:
        ops.append(np.sqrt(gamma_decay) * sm1)
        ops.append(np.sqrt(gamma_decay) * sm2)
    if gamma_phi > 0:
        ops.append(np.sqrt(gamma_phi) * n1)
        ops.append(np.sqrt(gamma_phi) * n2)
    return ops


## Noisy evolution and effective map

In [ ]:
def run_noisy_evolution(psi0, T, omega_max, alpha, V, delta0, p, q,
                        gamma_decay=0.0, gamma_phi=0.0, n_steps=220):
    H = build_time_dependent_hamiltonian(T, omega_max, alpha, V, delta0, p, q)
    times = np.linspace(0.0, T, n_steps)
    c_ops = collapse_ops(gamma_decay=gamma_decay, gamma_phi=gamma_phi)
    result = mesolve(H, psi0, times, c_ops)
    return result.states[-1]

def state_to_column_mixed(state):
    vals = []
    for basis_state in basis_states:
        if state.isket:
            amp = basis_state.overlap(state)
            vals.append(amp)
        else:
            val = (basis_state.dag() * state * basis_state)
            vals.append(np.real(val.full()[0, 0]) if hasattr(val, 'full') else np.real(val))
    return np.array(vals, dtype=complex)

def build_noisy_effective_map(T, omega_max, alpha, V, delta0, p, q,
                              gamma_decay=0.0, gamma_phi=0.0, n_steps=220):
    cols = []
    finals = []
    for psi0 in basis_states:
        final_state = run_noisy_evolution(
            psi0, T, omega_max, alpha, V, delta0, p, q,
            gamma_decay=gamma_decay, gamma_phi=gamma_phi, n_steps=n_steps
        )
        cols.append(state_to_column_mixed(final_state))
        finals.append(final_state)
    return np.column_stack(cols), finals


## Diagnostics

In [ ]:
def process_fidelity(U_eff, U_target):
    d = U_target.shape[0]
    return float(np.abs(np.trace(np.conjugate(U_target.T) @ U_eff)) ** 2 / (d ** 2))

def wrapped_phase(x):
    return np.angle(np.exp(1j * x))

def diagonal_phases(U_eff):
    return np.angle(np.diag(U_eff))

def entangling_phase(U_eff):
    phi = diagonal_phases(U_eff)
    return wrapped_phase(phi[0] + phi[3] - phi[1] - phi[2])

def extract_local_z_phases(U_eff):
    phi00, phi01, phi10, phi11 = diagonal_phases(U_eff)
    phi_ent = entangling_phase(U_eff)
    a = wrapped_phase(phi10 - phi00)
    b = wrapped_phase(phi01 - phi00)
    global_phase = phi00
    return global_phase, a, b, phi_ent

def local_z_diagonal(a, b):
    return np.diag([1.0, np.exp(1j*b), np.exp(1j*a), np.exp(1j*(a+b))]).astype(complex)

def compensate_local_z(U_eff):
    global_phase, a, b, phi_ent = extract_local_z_phases(U_eff)
    U1 = np.exp(-1j * global_phase) * U_eff
    U2 = U1 @ local_z_diagonal(-a, -b)
    return U2, global_phase, a, b, phi_ent

def compensated_cz_fidelity(U_eff):
    U_comp, _, _, _, _ = compensate_local_z(U_eff)
    return process_fidelity(U_comp, U_cz)

def leakage_from_finals(final_states):
    scores = []
    for idx, final_state in enumerate(final_states):
        basis_state = basis_states[idx]
        if final_state.isket:
            amp = basis_state.overlap(final_state)
            score = np.abs(amp) ** 2
        else:
            val = (basis_state.dag() * final_state * basis_state)
            score = np.real(val.full()[0, 0]) if hasattr(val, 'full') else np.real(val)
        scores.append(score)
    return float(1.0 - np.mean(scores))

def coherence_proxy(final_states):
    vals = []
    for state in final_states:
        rho = state.proj() if state.isket else state
        arr = rho.full()
        off = arr.copy()
        np.fill_diagonal(off, 0.0)
        vals.append(np.mean(np.abs(off)))
    return float(np.mean(vals))


## Noise scan and balanced order parameter

In [ ]:
gamma_decay_vals = np.linspace(0.0, 0.12, 25)
gamma_phi_vals = np.linspace(0.0, 0.12, 25)

cz_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))
coh_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))
leak_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))

for i, gamma_phi in enumerate(gamma_phi_vals):
    for j, gamma_decay in enumerate(gamma_decay_vals):
        U_eff, finals = build_noisy_effective_map(
            opt['T'], opt['omega_max'], opt['alpha'], V, opt['delta0'], opt['p'], opt['q'],
            gamma_decay=gamma_decay, gamma_phi=gamma_phi, n_steps=200
        )
        cz_map[i, j] = compensated_cz_fidelity(U_eff)
        coh_map[i, j] = coherence_proxy(finals)
        leak_map[i, j] = leakage_from_finals(finals)

S_true = cz_map * coh_map * (1.0 - leak_map)
S_true_norm = S_true / np.max(S_true)
print("max normalized S_true =", S_true_norm.max())


## Fit the effective coordinate

In [ ]:
S_gamma = S_true_norm[0, :]
S_phi = S_true_norm[:, 0]

interp_gamma = interp1d(gamma_decay_vals, S_gamma, kind='linear', fill_value='extrapolate')

def collapse_loss(lam):
    gamma_eff_phi = lam * gamma_phi_vals
    pred = interp_gamma(np.clip(gamma_eff_phi, gamma_decay_vals.min(), gamma_decay_vals.max()))
    return float(np.mean((pred - S_phi) ** 2))

fit = minimize_scalar(collapse_loss, bounds=(0.1, 5.0), method='bounded')
lambda_fit = float(fit.x)

print("best-fit lambda =", lambda_fit)
print("axis-slice collapse loss =", fit.fun)


## Build a 1D predictor S_hat(gamma_eff)

In [ ]:
gamma_eff_grid = np.zeros_like(S_true_norm)
for i, gamma_phi in enumerate(gamma_phi_vals):
    for j, gamma_decay in enumerate(gamma_decay_vals):
        gamma_eff_grid[i, j] = gamma_decay + lambda_fit * gamma_phi

# Use the full cloud to estimate a 1D response curve by binning in gamma_eff.
gamma_eff_flat = gamma_eff_grid.ravel()
S_flat = S_true_norm.ravel()

order = np.argsort(gamma_eff_flat)
gamma_eff_sorted = gamma_eff_flat[order]
S_sorted = S_flat[order]

n_bins = 24
bins = np.linspace(gamma_eff_sorted.min(), gamma_eff_sorted.max(), n_bins + 1)
bin_centers = 0.5 * (bins[:-1] + bins[1:])
bin_means = np.full(n_bins, np.nan)

for k in range(n_bins):
    mask = (gamma_eff_sorted >= bins[k]) & (gamma_eff_sorted < bins[k+1])
    if np.any(mask):
        bin_means[k] = np.mean(S_sorted[mask])

valid = ~np.isnan(bin_means)
S_hat_interp = interp1d(bin_centers[valid], bin_means[valid], kind='linear', fill_value='extrapolate')

S_pred_flat = S_hat_interp(gamma_eff_flat)
S_pred = S_pred_flat.reshape(S_true_norm.shape)
S_pred = np.clip(S_pred, 0.0, 1.0)


## Prediction quality metrics

In [ ]:
residual_map = S_true_norm - S_pred
abs_residual_map = np.abs(residual_map)

mse = float(np.mean((S_true_norm - S_pred) ** 2))
mae = float(np.mean(np.abs(S_true_norm - S_pred)))
var_true = float(np.var(S_true_norm))
r2 = float(1.0 - mse / var_true) if var_true > 0 else np.nan

# local low-noise scores
low_noise_mask = gamma_eff_grid <= 0.05
mse_low = float(np.mean((S_true_norm[low_noise_mask] - S_pred[low_noise_mask]) ** 2))
mae_low = float(np.mean(np.abs(S_true_norm[low_noise_mask] - S_pred[low_noise_mask])))

print("Prediction metrics:")
print("MSE =", mse)
print("MAE =", mae)
print("R^2 =", r2)
print("Low-noise MSE =", mse_low)
print("Low-noise MAE =", mae_low)


## Effective-coordinate response curve

In [ ]:
plt.figure(figsize=(7.2, 5.0))
plt.scatter(gamma_eff_flat, S_flat, s=12, alpha=0.25, label='2D samples')
plt.plot(bin_centers, bin_means, linewidth=2.2, label='1D predictor')
plt.xlabel('gamma_eff = gamma + lambda * gamma_phi')
plt.ylabel('Normalized S')
plt.title('Learned 1D phase-lock response curve')
plt.legend()
plt.tight_layout()
plt.show()


## Truth vs prediction

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

im0 = axes[0].imshow(
    S_true_norm,
    origin='lower',
    aspect='auto',
    extent=[gamma_decay_vals[0], gamma_decay_vals[-1], gamma_phi_vals[0], gamma_phi_vals[-1]],
    vmin=0, vmax=1
)
axes[0].set_title('True 2D phase-lock map')
axes[0].set_xlabel('gamma')
axes[0].set_ylabel('gamma_phi')

im1 = axes[1].imshow(
    S_pred,
    origin='lower',
    aspect='auto',
    extent=[gamma_decay_vals[0], gamma_decay_vals[-1], gamma_phi_vals[0], gamma_phi_vals[-1]],
    vmin=0, vmax=1
)
axes[1].set_title('Predicted from gamma_eff only')
axes[1].set_xlabel('gamma')
axes[1].set_ylabel('gamma_phi')

fig.colorbar(im0, ax=axes[0], label='normalized S')
fig.colorbar(im1, ax=axes[1], label='normalized S')
plt.tight_layout()
plt.show()


## Residual maps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

im0 = axes[0].imshow(
    residual_map,
    origin='lower',
    aspect='auto',
    extent=[gamma_decay_vals[0], gamma_decay_vals[-1], gamma_phi_vals[0], gamma_phi_vals[-1]]
)
axes[0].set_title('Signed residual: true - predicted')
axes[0].set_xlabel('gamma')
axes[0].set_ylabel('gamma_phi')

im1 = axes[1].imshow(
    abs_residual_map,
    origin='lower',
    aspect='auto',
    extent=[gamma_decay_vals[0], gamma_decay_vals[-1], gamma_phi_vals[0], gamma_phi_vals[-1]]
)
axes[1].set_title('Absolute residual')
axes[1].set_xlabel('gamma')
axes[1].set_ylabel('gamma_phi')

fig.colorbar(im0, ax=axes[0], label='signed error')
fig.colorbar(im1, ax=axes[1], label='absolute error')
plt.tight_layout()
plt.show()


## Axis-slice prediction check

In [ ]:
S_gamma_pred = S_pred[0, :]
S_phi_pred = S_pred[:, 0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))

axes[0].plot(gamma_decay_vals, S_gamma, label='true')
axes[0].plot(gamma_decay_vals, S_gamma_pred, label='predicted')
axes[0].set_xlabel('gamma')
axes[0].set_ylabel('Normalized S')
axes[0].set_title('gamma_phi = 0 slice')
axes[0].legend()

axes[1].plot(gamma_phi_vals, S_phi, label='true')
axes[1].plot(gamma_phi_vals, S_phi_pred, label='predicted')
axes[1].set_xlabel('gamma_phi')
axes[1].set_ylabel('Normalized S')
axes[1].set_title('gamma = 0 slice')
axes[1].legend()

plt.tight_layout()
plt.show()


## Compact summary

In [ ]:
summary_text = f'''
Predicting phase-lock quality from one effective noise coordinate

Protocol:
T      = {opt["T"]:.4f}
alpha  = {opt["alpha"]:.4f}
Omega  = {opt["omega_max"]:.4f}
Delta0 = {opt["delta0"]:.4f}
p      = {opt["p"]:.4f}
q      = {opt["q"]:.4f}

Balanced order parameter:
S = F_CZ * C * (1 - L)

Effective coordinate:
gamma_eff = gamma + lambda * gamma_phi
lambda = {lambda_fit:.4f}

Prediction quality:
global MSE     = {mse:.6e}
global MAE     = {mae:.6e}
global R^2     = {r2:.6f}
low-noise MSE  = {mse_low:.6e}
low-noise MAE  = {mae_low:.6e}

Interpretation:
- the 1D predictor captures a substantial part of the 2D phase-lock structure,
- prediction is typically best in the low-noise regime where the gate is most usable,
- deviations from the predictor reveal the genuinely irreducible 2D character of the full noisy channel.
'''
print(summary_text)


## Final conclusion

This notebook upgrades the effective-noise-coordinate idea from a visual collapse to a predictive model.

Instead of only observing that the balanced phase-lock map roughly organizes along:

`gamma_eff = gamma + lambda * gamma_phi`

it now builds a 1D predictor and measures how accurately that predictor reconstructs the full 2D map.

That gives the cleanest final interpretation:

**the noisy phase-locked CZ regime is not only approximately organized by a one-parameter effective decoherence coordinate, but can be partially predicted from that coordinate alone, with the remaining residuals quantifying the irreducible two-dimensional structure of the model.**
